# Milestone 11 - Vet a use case with MLflow

The **Use case and fit** lessons review an AI idea with a model: "Ask the model to draft the
review" and "Challenge my use case". This notebook builds the machinery that keeps those reviews
honest, with **MLflow's real classes and functions**, the same ones the app's `drill review`
commands use:

| Step | What you use |
|---|---|
| 1. Trace a review | `@mlflow.trace`, `SpanType`, `mlflow.update_current_trace` |
| 2. Checks by code | `mlflow.log_feedback`, `AssessmentSource(source_type="CODE")` |
| 3. Your label | `mlflow.log_feedback` with a `HUMAN` source |
| 4. Read labels back | `mlflow.search_traces`, `trace.info.assessments` |
| 5. A scorer and an eval | `@scorer`, `Feedback`, `mlflow.genai.evaluate` |
| 6. Does a judge agree with you? | `sklearn.metrics.cohen_kappa_score` |
| 7. The prompt registry | `mlflow.genai.register_prompt`, `set_prompt_alias`, `load_prompt` |

Lessons it builds on: *A well-architected review*, *Checking your LLM judge*, *Agreement beyond
chance* and *The prompt registry*.

**How to run it:** `uv run jupyter lab notebooks/m11_vet_a_use_case.ipynb`. Fill in each
`raise NotImplementedError`, then run the test cell under it. MLflow writes to
`.drill/m11/mlflow.db`; see what you made with
`uv run mlflow ui --backend-store-uri sqlite:///.drill/m11/mlflow.db`.

In [ ]:
import os
import tempfile
from pathlib import Path

import ipytest
import mlflow
import pytest
from mlflow.entities import AssessmentSource, Feedback, SpanType
from mlflow.genai.scorers import scorer
from sklearn.metrics import cohen_kappa_score

from drill.content import review
from drill.review_checks import flags_gap, review_checks

ipytest.autoconfig(clean="test_*")
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"

# Where MLflow keeps what you make. The tests use a fresh temporary folder each run.
if os.getenv("DRILL_NOTEBOOK_OFFLINE") == "1":
    folder = Path(tempfile.mkdtemp(prefix="drill-m11-"))
else:
    folder = Path(".drill/m11")
    folder.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri(f"sqlite:///{folder / 'mlflow.db'}")
mlflow.set_experiment("m11-vet-a-use-case")

# The HR assistant's card from the lessons, with the owner left blank on purpose.
CARD = {key: example for key, _label, example in review.CARD_FIELDS}
CARD["owner"] = ""

LIVE = False   # True: reviews come from your local Ollama, through LangChain's ChatOpenAI


def scripted_model(prompt: str) -> str:
    """A stand-in model, so the notebook gives the same result on every machine.
    It flags every card line marked 'not written', then names each pillar."""
    blank = [line[2:].split(":")[0] for line in prompt.splitlines() if line.endswith(": not written")]
    gaps = "\n".join(f"- Unknown: who can confirm the {b}?" for b in blank) or "- none"
    return "Gaps:\n" + gaps + "\n\n" + "\n".join(f"**{p}**\nRisk: medium." for p in review.PILLARS)


if LIVE:
    from langchain_openai import ChatOpenAI   # Ollama speaks the same protocol as OpenAI

    chat = ChatOpenAI(base_url="http://localhost:11434/v1", api_key="ollama", model="gemma3:12b", temperature=0)
    model = lambda prompt: chat.invoke(prompt).content
else:
    model = scripted_model

print(review.review_prompt(CARD)[:400], "...")

## 1. Trace a review

A **trace** is MLflow's record of one request: what went in, what came out, how long it took,
and every step inside it (each step is a **span**). Decorating a function with
`@mlflow.trace` records a trace every time it runs, with the function's arguments as inputs
and its return value as the output.

**Tags** label the whole trace, so you can find it again. Tag each review with its prompt
version: when you change the prompt later, you can compare reviews from before and after.

Write `traced_review(card, model)`:

- decorate it with `@mlflow.trace(name="use_case_review", span_type=SpanType.CHAIN)`,
- inside, tag the trace with `mlflow.update_current_trace(tags={...})`: `"feature"` set to
  `"use_case_review"` and `"prompt_version"` set to `review.REVIEW_VERSION`,
- build the prompt with `review.review_prompt(card)`, send it to `model`, and return the text.

In [ ]:
# TODO: put the @mlflow.trace decorator on the line above def.
def traced_review(card, model):
    # Step 1: tag the trace with the feature and the prompt version.
    # Step 2: build the prompt from the card, send it to model, return the reply.
    raise NotImplementedError

In [ ]:
%%ipytest

def test_a_review_is_traced_with_its_prompt_version():
    text = traced_review(CARD, scripted_model)
    mlflow.flush_trace_async_logging()   # traces are saved in the background: wait for it
    trace = mlflow.get_trace(mlflow.get_last_active_trace_id())
    assert trace.info.tags["prompt_version"] == review.REVIEW_VERSION
    assert trace.info.tags["feature"] == "use_case_review"
    root = trace.data.spans[0]
    assert root.name == "use_case_review" and root.span_type == SpanType.CHAIN
    assert root.inputs["card"] == CARD and root.outputs == text

## 2. Checks by code, as feedback

Some rules can be checked without a model: does the review name every pillar, does it decide
for the reader, does it say Unknown when the card has blank lines? `review_checks(text, blank)`
returns those checks, `True` meaning the rule was **broken**.

MLflow stores a judgement about a trace as **feedback**, an *assessment* attached to the trace.
Each one has a name, a value and a **source**: who made the judgement. Code is
`AssessmentSource(source_type="CODE", source_id="review_checks")`.

Write `log_code_checks(trace_id, text, blank)`: for every check whose value is `True` or
`False` (skip the count), call `mlflow.log_feedback(trace_id=..., name=..., value=..., source=...)`.

In [ ]:
def log_code_checks(trace_id, text, blank):
    # Step 1: run review_checks(text, blank).
    # Step 2: for each name and value that is a bool, log it as feedback from CODE.
    raise NotImplementedError

In [ ]:
%%ipytest

def test_each_check_lands_on_the_trace_from_code():
    text = traced_review(CARD, scripted_model)
    mlflow.flush_trace_async_logging()
    trace_id = mlflow.get_last_active_trace_id()
    log_code_checks(trace_id, text, review.blank_fields(CARD))
    got = {a.name: a for a in mlflow.get_trace(trace_id).info.assessments}
    assert {"missing_pillar", "decides_for_them", "no_unknowns_for_blanks"} <= set(got)
    assert "pillars_named" not in got, "the count is not a check"
    assert got["missing_pillar"].value is False and got["missing_pillar"].source.source_type == "CODE"

## 3. Your label

Code can't tell whether a review **invented** a fact. You can. Your judgement is feedback too,
from a `HUMAN` source. In the app, `drill review label` asks you, one review at a time.

Write `log_label(trace_id, grounded, note="")`: log feedback named `"grounded"` with the value
`grounded`, your `note` as the `rationale` (or `None` when it's empty), and
`AssessmentSource(source_type="HUMAN", source_id="you")`.

In [ ]:
def log_label(trace_id, grounded, note=""):
    raise NotImplementedError

In [ ]:
%%ipytest

def test_your_label_is_human_feedback():
    traced_review(CARD, scripted_model)
    mlflow.flush_trace_async_logging()
    trace_id = mlflow.get_last_active_trace_id()
    log_label(trace_id, False, "it named a tool the card never mentions")
    label = [a for a in mlflow.get_trace(trace_id).info.assessments if a.name == "grounded"][0]
    assert label.value is False and label.source.source_type == "HUMAN"
    assert label.rationale == "it named a tool the card never mentions"

## 4. Read the labels back

`mlflow.search_traces` finds traces by their tags, with a filter written like SQL:
`filter_string="tags.feature = 'use_case_review'"`. With `return_type="list"` you get `Trace`
objects, and each one's `trace.info.assessments` holds its feedback.

Write `read_labels(name="grounded", source_type="HUMAN")`: return a dict from trace id to the
value of that feedback, for every review trace that has it.

In [ ]:
def read_labels(name="grounded", source_type="HUMAN"):
    # Step 1: search the traces tagged feature = 'use_case_review'.
    # Step 2: for each, look through trace.info.assessments for the name and source type.
    raise NotImplementedError

In [ ]:
%%ipytest

def test_labels_come_back_by_trace():
    traced_review(CARD, scripted_model)
    mlflow.flush_trace_async_logging()
    trace_id = mlflow.get_last_active_trace_id()
    log_label(trace_id, True)
    labels = read_labels()
    assert labels[trace_id] is True
    assert all(isinstance(v, bool) for v in labels.values())

## 5. A scorer and an evaluation

An **eval** runs the prompt on cases with known answers and scores every reply. Here each case
is a card with a planted gap, and the right answer is a review that flags it.

- A **scorer** is a function decorated with `@scorer`. MLflow calls it with the case's
  `inputs`, the reply (`outputs`) and the case's `expectations`, and it returns a `Feedback`
  (a value and a rationale) that is saved on the reply's trace.
- `mlflow.genai.evaluate(data=..., predict_fn=..., scorers=[...])` runs `predict_fn` on each
  case's inputs, scores the replies, and returns a result whose `.metrics` holds the averages.

Write two things:

- `gaps_flagged(inputs, outputs, expectations)`: for each gap in `expectations["gaps"]`, check
  `flags_gap(outputs, [gap])`. Return `Feedback(value=..., rationale=...)`: `True` when none
  were missed, and a rationale naming any that were.
- `run_eval(model)`: evaluate `traced_review` with `model` on `EVAL_DATA` (below) with your
  scorer, and return the result's `metrics`.

In [ ]:
EVAL_DATA = [
    {"inputs": {"card": {**CARD, "owner": ""}}, "expectations": {"gaps": ["owner"]}},
    {"inputs": {"card": {**CARD, "owner": "Head of HR", "data": ""}}, "expectations": {"gaps": ["data"]}},
    {"inputs": {"card": {**CARD, "owner": "", "feedback": ""}}, "expectations": {"gaps": ["owner", "feedback"]}},
]

In [ ]:
# TODO: decorate this one with @scorer.
def gaps_flagged(inputs, outputs, expectations):
    raise NotImplementedError


def run_eval(model):
    # predict_fn gets each case's inputs as keyword arguments: here, card=...
    raise NotImplementedError

In [ ]:
%%ipytest

def test_the_scorer_flags_a_miss():
    good = gaps_flagged(inputs={}, outputs="Unknown: who is the owner?", expectations={"gaps": ["owner"]})
    bad = gaps_flagged(inputs={}, outputs="All fine.", expectations={"gaps": ["owner"]})
    assert good.value is True and bad.value is False and "owner" in bad.rationale


def test_the_eval_scores_every_case():
    metrics = run_eval(scripted_model)
    assert metrics["gaps_flagged/mean"] == 1.0

## 6. Does a judge agree with you?

A model can judge reviews too ("did this invent a fact?"), but only once you know it agrees
with you. **Cohen's kappa** is agreement beyond what chance would give: 1 is perfect, 0 is no
better than guessing. scikit-learn computes it: `cohen_kappa_score(yours, the_judges)`.

When every label is the same, chance agreement is total and kappa can't be worked out (it comes
out as `nan`). Write `judge_agreement(human, judge)`: return `None` in that case, else the kappa
rounded to 2 places.

In [ ]:
def judge_agreement(human, judge):
    # Step 1: if all the labels, yours and the judge's together, are the same, return None.
    # Step 2: otherwise return round(cohen_kappa_score(human, judge), 2).
    raise NotImplementedError

In [ ]:
%%ipytest

def test_kappa_and_the_no_variety_case():
    assert judge_agreement([True, True, False, False], [True, False, False, False]) == 0.5
    assert judge_agreement([True, False], [True, False]) == 1.0
    assert judge_agreement([True, True, True], [True, True, True]) is None

## 7. The prompt registry

The **prompt registry** keeps every version of a prompt, so you can see what changed, roll
back, and tag the one in use with an **alias** like `production`. A registered prompt is a
**template**: `{{card}}` marks where the card goes.

- `mlflow.genai.register_prompt(name=..., template=..., commit_message=...)` adds a version and
  returns it (its number is `.version`).
- `mlflow.genai.set_prompt_alias(name, "production", version)` points the alias at it.
- `mlflow.genai.load_prompt("prompts:/<name>@production")` loads it back, and its
  `.format(card=...)` fills in the template.

Write `register_review_prompt()`, which registers `review.review_template()` as
`"drill-use-case-review"` with `review.REVIEW_VERSION` as the commit message, points
`production` at it and returns the version number. Then write `production_prompt(card)`,
which loads the production version and fills in `review.card_lines(card)`.

In [ ]:
def register_review_prompt():
    raise NotImplementedError


def production_prompt(card):
    raise NotImplementedError

In [ ]:
%%ipytest

def test_the_registered_prompt_is_the_prompt_the_page_sends():
    version = register_review_prompt()
    assert isinstance(version, int) and version >= 1
    assert production_prompt(CARD) == review.review_prompt(CARD)

## What you built, and where it lives in the app

- The page sends each review with its prompt version and an id; the drill server traces it, as
  in step 1 (`src/drill/server/observe.py`), and logs the code checks as in step 2.
- "Was this useful?" under each draft is feedback from a person, as in step 3.
- `drill review label` and `drill review calibrate` are steps 3, 4 and 6
  (`src/drill/review_ops.py`); `drill review eval` is step 5 (`src/drill/review_eval.py`);
  `drill review register` is step 7.

Open the MLflow UI and look at one trace: its spans, its tags, and the feedback from code and
from you, side by side.